#  Notebook 2 : Préparation et nettoyage des données

## Objectif : Nettoyer, normaliser et fusionner  5 fichiers sources pour créer des tables prêtes pour Power BI.

### Étape 1 : Import des bibliothèques et chargement des données

In [1]:
import pandas as pd
import numpy as np

# Configuration pour afficher toutes les colonnes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Chargement des fichiers CSV (à adapter selon vos chemins)
population = pd.read_csv("data/raw/Population.csv", sep=",")
regions = pd.read_csv("data/raw/RegionCountry.csv", sep=",")
stabilite_politique = pd.read_csv("data/raw/PoliticalStability.csv", sep=",")
mortalite_eau = pd.read_csv("data/raw/MortalityRateAttributedToWater.csv", sep=",")
services_eau = pd.read_csv("data/raw/BasicAndSafelyManagedDrinkingWaterServices.csv", sep=",")

print("✅ Fichiers chargés avec succès !")

✅ Fichiers chargés avec succès !


In [2]:
# Affichez les colonnes de chaque DataFrame
print("Population.csv :", list(population.columns))
print("RegionCountry.csv :", list(regions.columns))
print("PoliticalStability.csv :", list(stabilite_politique.columns))
print("MortalityRateAttributedToWater.csv :", list(mortalite_eau.columns))
print("BasicAndSafelyManagedDrinkingWaterServices.csv :", list(services_eau.columns))

Population.csv : ['Country', 'Granularity', 'Year', 'Population']
RegionCountry.csv : ['REGION (DISPLAY)', 'COUNTRY (DISPLAY)']
PoliticalStability.csv : ['Country', 'Year', 'Political_Stability', 'Granularity']
MortalityRateAttributedToWater.csv : ['Year', 'Country', 'Granularity', 'Mortality rate attributed to exposure to unsafe WASH services', 'WASH deaths']
BasicAndSafelyManagedDrinkingWaterServices.csv : ['Year', 'Country', 'Granularity', 'Population using at least basic drinking-water services (%)', 'Population using safely managed drinking-water services (%)']


###  Étape 2 : Nettoyage et standardisation des noms de pays

Problème identifié : Certains pays n’ont pas de région associée ou ont des noms différents selon les fichiers.

In [3]:
# --- 2. Nettoyage et standardisation ---

# Fonction pour normaliser les noms de pays
def normaliser_texte(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    text = " ".join(text.split())
    return text

# Dictionnaire pour renommer les pays
COUNTRY_RENAMES = {
    "North Macedonia": "Republic of North Macedonia",
    # Ajoutez ici d'autres renommages si nécessaire
}

# Dictionnaire pour associer les pays à des régions manuellement
REGION_OVERRIDES = {
    "American Samoa": "Western Pacific",
    "Anguilla": "Americas",
    "Aruba": "Americas",
    "Bermuda": "Americas",
    "Bonaire, Sint Eustatius and Saba": "Americas",
    "British Virgin Islands": "Americas",
    "Cayman Islands": "Americas",
    "Channel Islands": "Europe",
    "China, Hong Kong SAR": "Western Pacific",
    "China, Macao SAR": "Western Pacific",
    "China, Taiwan Province of": "Western Pacific",
    "China, mainland": "Western Pacific",
    "Curaçao": "Americas",
    "Falkland Islands (Malvinas)": "Americas",
    "Faroe Islands": "Europe",
    "French Guiana": "Americas",
    "French Polynesia": "Western Pacific",
    "Gibraltar": "Europe",
    "Greenland": "Americas",
    "Guadeloupe": "Americas",
    "Guam": "Western Pacific",
    "Holy See": "Europe",
    "Isle of Man": "Europe",
    "Liechtenstein": "Europe",
    "Martinique": "Americas",
    "Mayotte": "Africa",
    "Montserrat": "Americas",
    "Netherlands Antilles (former)": "Americas",
    "New Caledonia": "Western Pacific",
    "Northern Mariana Islands": "Western Pacific",
    "Palestine": "Eastern Mediterranean",
    "Puerto Rico": "Americas",
    "Réunion": "Africa",
    "Saint Barthélemy": "Americas",
    "Saint Helena, Ascension and Tristan da Cunha": "Africa",
    "Saint Pierre and Miquelon": "Americas",
    "Saint-Martin (French part)": "Americas",
    "Serbia and Montenegro": "Europe",
    "Sint Maarten (Dutch part)": "Americas",
    "Sint Maarten  (Dutch part)": "Americas",
    "Sudan (former)": "Eastern Mediterranean",
    "Tokelau": "Western Pacific",
    "Turks and Caicos Islands": "Americas",
    "United States Virgin Islands": "Americas",
    "Wallis and Futuna Islands": "Western Pacific",
    "Western Sahara": "Africa",
}

# Fonction pour standardiser les noms de pays (suppression des accents, espaces, etc.)
def standardiser_nom_pays(nom):
    if pd.isna(nom):
        return np.nan
    nom = str(nom).strip().lower()
    # Dictionnaire de renommage pour les pays problématiques
    renommages = {
        "north macedonia": "republic of north macedonia",
        "côte d'ivoire": "cote divoire",
        "united states": "united states of america",
    }
    return renommages.get(nom, nom)

# Appliquer la standardisation à toutes les colonnes 'Country'
for df in [population, regions, stabilite_politique, mortalite_eau, services_eau]:
    if 'Country' in df.columns:
        df['Country'] = df['Country'].apply(standardiser_nom_pays)
    elif 'COUNTRY (DISPLAY)' in df.columns:
        df['COUNTRY (DISPLAY)'] = df['COUNTRY (DISPLAY)'].apply(standardiser_nom_pays)

print("✅ Noms de pays standardisés !")

✅ Noms de pays standardisés !


###  Étape 3 : Création des tables dimensionnelles

Objectif : Créer des tables de référence pour Power BI (pays, régions, années, indicateurs).

#### Table dimensionnelle des pays et régions

In [4]:
# Table des pays et régions (à partir de RegionCountry.csv)
# Renommer les colonnes pour simplifier
regions = regions.rename(columns={
    'COUNTRY (DISPLAY)': 'Country',
    'REGION (DISPLAY)': 'Region'
})
# Créer dim_pays (pays + région)
dim_pays = regions[['Country', 'Region']].drop_duplicates().copy()
# Ajouter manuellement les pays manquants (exemple : American Samoa → Western Pacific)
pays_manquants = {
    "American Samoa": "Western Pacific",
    "Anguilla": "Americas",
    "Aruba": "Americas",
    "Bermuda": "Americas",
    "Bonaire, Sint Eustatius and Saba": "Americas",
    "British Virgin Islands": "Americas",
    "Cayman Islands": "Americas",
    "Channel Islands": "Europe",
    "China, Hong Kong SAR": "Western Pacific",
    "China, Macao SAR": "Western Pacific",
    "China, Taiwan Province of": "Western Pacific",
    "China, mainland": "Western Pacific",
    "Curaçao": "Americas",
    "Falkland Islands (Malvinas)": "Americas",
    "Faroe Islands": "Europe",
    "French Guyana": "Americas",
    "French Polynesia": "Western Pacific",
    "Gibraltar": "Europe",
    "Greenland": "Americas",
    "Guadeloupe": "Americas",
    "Guam": "Western Pacific",
    "Holy See": "Europe",
    "Isle of Man": "Europe",
    "Liechtenstein": "Europe",
    "Martinique": "Americas",
    "Mayotte": "Africa",
    "Montserrat": "Americas",
    "Netherlands Antilles (former)": "Americas",
    "New Caledonia": "Western Pacific",
    "Northern Mariana Islands": "Western Pacific",
    "Palestine": "Eastern Mediterranean",
    "Puerto Rico": "Americas",
    "Réunion": "Africa",
    "Saint Barthélemy": "Americas",
    "Saint Helena, Ascension and Tristan da Cunha": "Africa",
    "Saint Pierre and Miquelon": "Americas",
    "Saint-Martin (French part)": "Americas",
    "Serbia and Montenegro": "Europe",
    "Sint Maarten (Dutch part)": "Americas",
    "Sint Maarten  (Dutch part)": "Americas",
    "Sudan (former)": "Eastern Mediterranean",
    "Tokelau": "Western Pacific",
    "Turks and Caicos Islands": "Americas",
    "United States Virgin Islands": "Americas",
    "Wallis and Futuna Islands": "Western Pacific",
    "Western Sahara": "Africa",
    
}

# Ajouter les pays manquants à dim_pays
for pays, region in pays_manquants.items():
    if not dim_pays[dim_pays['Country'] == pays].empty:
        continue
    dim_pays = pd.concat([dim_pays, pd.DataFrame({'Country': [pays], 'Region': [region]})], ignore_index=True)

print(f"✅ Table dim_pays créée : {len(dim_pays)} pays uniques.")

✅ Table dim_pays créée : 240 pays uniques.


Lien URL pour l'enrichissement des pays : https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes/blob/master/all/all.csv ou fichier country_region_reference_fr.csv pour créer la table de référence des pays et régions.

## Enrichissement de la table dim_pays avec des données ISO3 , continent, sous-région et région OMS

#### Étape 1 : Charger le fichier de référence

In [5]:
# Charger le fichier de référence (remplacez le chemin si nécessaire)
reference_pays = pd.read_csv("data/processed/country_region_reference_fr.csv", sep=";")

# Afficher les premières lignes pour vérifier
print(reference_pays.head())

           pays  region iso3 continent  sous_region_onu pays_source_en
0       Algérie  Africa  DZA    Africa  Northern Africa        Algeria
1        Angola  Africa  AGO    Africa    Middle Africa         Angola
2         Bénin  Africa  BEN    Africa   Western Africa          Benin
3      Botswana  Africa  BWA    Africa  Southern Africa       Botswana
4  Burkina Faso  Africa  BFA    Africa   Western Africa   Burkina Faso


#### Étape 2 : Nettoyer et standardiser les noms de pays

In [6]:
# Fonction pour standardiser les noms (identique à celle utilisée précédemment)
def standardiser_nom_pays(nom):
    if pd.isna(nom):
        return np.nan
    nom = str(nom).strip().lower()
    renommages = {
        "north macedonia": "republic of north macedonia",
        "côte d'ivoire": "cote divoire",
        "united states": "united states of america",
    }
    return renommages.get(nom, nom)

# Appliquer la standardisation à la colonne 'pays' de reference_pays
reference_pays['pays_clean'] = reference_pays['pays'].apply(standardiser_nom_pays)
reference_pays['pays_source_en_clean'] = reference_pays['pays_source_en'].apply(standardiser_nom_pays)

#### Étape 3 :  Fusionner dim_pays avec reference_pays

In [7]:
# Standardiser les noms dans dim_pays (si ce n'est pas déjà fait)
dim_pays['Country_clean'] = dim_pays['Country'].apply(standardiser_nom_pays)

# Fusionner avec reference_pays sur 'pays_clean'
dim_pays_enriched = dim_pays.merge(
    reference_pays,
    left_on='Country_clean',
    right_on='pays_clean',
    how='left'
)

# Si certains pays ne sont pas trouvés, essayer avec 'pays_source_en_clean'
pays_non_trouves = dim_pays_enriched[dim_pays_enriched['pays'].isna()]
if not pays_non_trouves.empty:
    # Fusionner à nouveau sur 'pays_source_en_clean' pour les pays restants
    dim_pays_restants = dim_pays[dim_pays['Country_clean'].isin(pays_non_trouves['Country_clean'])]
    dim_pays_restants_enriched = dim_pays_restants.merge(
        reference_pays,
        left_on='Country_clean',
        right_on='pays_source_en_clean',
        how='left'
    )
    # Remplacer les lignes non trouvées dans dim_pays_enriched
    dim_pays_enriched.update(dim_pays_restants_enriched)

# Nettoyer les colonnes inutiles
dim_pays_enriched = dim_pays_enriched.drop(columns=['pays_clean', 'pays_source_en_clean', 'Country_clean'], errors='ignore')

# Renommer les colonnes pour plus de clarté
dim_pays_enriched = dim_pays_enriched.rename(columns={
    'pays': 'pays_fr',
    'pays_source_en': 'pays_en',
    'iso3': 'code_iso3',
    'region': 'region_oms',
})

# Conserver uniquement les colonnes utiles
colonnes_utilisees = ['Country', 'Region', 'pays_fr', 'pays_en', 'code_iso3', 'region_oms', 'continent', 'sous_region_onu']
dim_pays_enriched = dim_pays_enriched[colonnes_utilisees]

print("✅ Table dim_pays enrichie avec succès !")
print(dim_pays_enriched.head())

✅ Table dim_pays enrichie avec succès !
     Country           Region    pays_fr    pays_en code_iso3  \
0    albania           Europe    Albanie    Albania       ALB   
1    andorra           Europe    Andorre    Andorra       AND   
2    armenia           Europe    Arménie    Armenia       ARM   
3  australia  Western Pacific  Australie  Australia       AUS   
4    austria           Europe   Autriche    Austria       AUT   

        region_oms continent            sous_region_onu  
0           Europe    Europe            Southern Europe  
1           Europe    Europe            Southern Europe  
2           Europe      Asia               Western Asia  
3  Western Pacific   Oceania  Australia and New Zealand  
4           Europe    Europe             Western Europe  


#### Étape 4 : Vérifier et corriger les valeurs manquantes

In [8]:
# Remplir les codes ISO3 manquants avec pycountry
import pycountry

def get_iso3(pays_en):
    try:
        country = pycountry.countries.search_fuzzy(pays_en)
        return country[0].alpha_3 if country else None
    except:
        return None

# Appliquer uniquement aux lignes où code_iso3 est vide
mask = dim_pays_enriched['code_iso3'].isna()
dim_pays_enriched.loc[mask, 'code_iso3'] = dim_pays_enriched.loc[mask, 'pays_en'].apply(get_iso3)

# Vérifier les valeurs manquantes
print("\nValeurs manquantes après enrichissement :")
print(dim_pays_enriched.isnull().sum())


Valeurs manquantes après enrichissement :
Country             0
Region              0
pays_fr            51
pays_en            51
code_iso3          51
region_oms         51
continent          51
sous_region_onu    51
dtype: int64


####  Étape 5 : Exporter la table enrichie

In [9]:
# Exporter la table enrichie
import os


os.makedirs("data/processed", exist_ok=True)
dim_pays_enriched.to_csv("data/processed/dim_pays_enriched.csv", index=False)

print("✅ Export de dim_pays_enriched.csv terminé !")

✅ Export de dim_pays_enriched.csv terminé !


#### Table dimensionnelle des années

In [10]:
# Extraire les années uniques de tous les fichiers
annees = set()
for df in [population, stabilite_politique, mortalite_eau, services_eau]:
    if 'Year' in df.columns:
        annees.update(df['Year'].dropna().unique())

dim_annee = pd.DataFrame({'Year': sorted(annees)})
print(f"✅ Table dim_annee créée : {len(dim_annee)} années uniques.")

✅ Table dim_annee créée : 19 années uniques.


#### Table dimensionnelle des indicateurs

In [11]:
# Liste des indicateurs et leurs unités
dim_indicateur = pd.DataFrame({
    'indicator': [
        'population_people',
        'political_stability',
        'basic_drinking_water_pct',
        'safely_managed_drinking_water_pct',
        'wash_mortality_rate_per_100k',
        'wash_deaths'
    ],
    'unit': [
        'people',
        'index',
        'percent',
        'percent',
        'per_100k',
        'people'
    ]
})
print("✅ Table dim_indicateur créée !")

✅ Table dim_indicateur créée !


## Étape 4 :  Préparation des bases pour chaque fichier

In [12]:
# Fonction pour préparer une base standardisée (pays, région, année)
def preparer_base(df, nom_df):
    df = df.copy()
    # Standardiser le nom du pays
    if 'Country' in df.columns:
        df['country'] = df['Country'].apply(standardiser_nom_pays)
    else:
        print(f"⚠️ Erreur : pas de colonne 'Country' dans {nom_df}")
        return pd.DataFrame()

    # Standardiser l'année
    if 'Year' in df.columns:
        df['year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
    else:
        print(f"⚠️ Erreur : pas de colonne 'Year' dans {nom_df}")
        return pd.DataFrame()

    # Ajouter la région depuis dim_pays
    df = df.merge(dim_pays, left_on='country', right_on='Country', how='left')
    df = df.drop(columns=['Country'], errors='ignore')  # Garder uniquement 'country' et 'Region'
    return df

# Préparer chaque fichier
population_base = preparer_base(population, "Population.csv")
stabilite_base = preparer_base(stabilite_politique, "PoliticalStability.csv")
mortalite_base = preparer_base(mortalite_eau, "MortalityRateAttributedToWater.csv")
services_base = preparer_base(services_eau, "BasicAndSafelyManagedDrinkingWaterServices.csv")

### Étape 5 : Construction de la table de faits

In [13]:
def construire_table_indicateur(base_df, mapping, source_name, scope_note):
    frames = []
    for col_source, meta in mapping.items():
        if col_source in base_df.columns:
            temp = base_df[['country', 'Region', 'year', col_source]].copy()
            temp = temp.rename(columns={col_source: 'value'})
            temp['indicator'] = meta['indicator']
            temp['unit'] = meta['unit']
            temp['source'] = source_name
            temp['scope_note'] = scope_note
            temp['value'] = pd.to_numeric(temp['value'], errors='coerce')
            frames.append(temp)
        else:
            print(f"⚠️ colonne {col_source} introuvable dans {source_name}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# Construire les tables d'indicateurs
population_long = construire_table_indicateur(
    population_base,
    {"Population": {"indicator": "population_people", "unit": "people"}},
    source_name="Population.csv",
    scope_note="Données démographiques. Valeur en nombre d'habitants."
)

stabilite_long = construire_table_indicateur(
    stabilite_base,
    {"Political_Stability": {"indicator": "political_stability", "unit": "index"}},
    source_name="PoliticalStability.csv",
    scope_note="Indicateur institutionnel."
)

services_long = construire_table_indicateur(
    services_base,
    {
        "Population using at least basic drinking-water services (%)": {
            "indicator": "basic_drinking_water_pct",
            "unit": "percent"
        },
        "Population using safely managed drinking-water services (%)": {
            "indicator": "safely_managed_drinking_water_pct",
            "unit": "percent"
        }
    },
    source_name="BasicAndSafelyManagedDrinkingWaterServices.csv",
    scope_note="Accès à l'eau potable."
)

mortalite_long = construire_table_indicateur(
    mortalite_base,
    {
        "Mortality rate attributed to exposure to unsafe WASH services": {
            "indicator": "wash_mortality_rate_per_100k",
            "unit": "per_100k"
        },
        "WASH deaths": {
            "indicator": "wash_deaths",
            "unit": "people"
        }
    },
    source_name="MortalityRateAttributedToWater.csv",
    scope_note="Proxy sanitaire WASH."
)

# Fusionner toutes les tables d'indicateurs
table_faits = pd.concat([population_long, stabilite_long, services_long, mortalite_long], ignore_index=True)
table_faits = table_faits.dropna(subset=['country', 'year', 'indicator'])

print(f"✅ Table de faits créée : {len(table_faits)} lignes")
print(table_faits.head())

✅ Table de faits créée : 46490 lignes
       country                 Region  year      value          indicator  \
0  afghanistan  Eastern Mediterranean  2000  20779.953  population_people   
1  afghanistan  Eastern Mediterranean  2000  10689.508  population_people   
2  afghanistan  Eastern Mediterranean  2000  10090.449  population_people   
3  afghanistan  Eastern Mediterranean  2000  15657.474  population_people   
4  afghanistan  Eastern Mediterranean  2000   4436.282  population_people   

     unit          source                                         scope_note  
0  people  Population.csv  Données démographiques. Valeur en nombre d'hab...  
1  people  Population.csv  Données démographiques. Valeur en nombre d'hab...  
2  people  Population.csv  Données démographiques. Valeur en nombre d'hab...  
3  people  Population.csv  Données démographiques. Valeur en nombre d'hab...  
4  people  Population.csv  Données démographiques. Valeur en nombre d'hab...  


### 5 - Enrichir dim_pays_enriched avec les données de référence des pays et régions

In [14]:
# Charger le fichier de référence
reference_pays = pd.read_csv("data/processed/country_region_reference_fr.csv", sep=";")

# Standardiser les noms de pays dans reference_pays
reference_pays['pays_clean'] = reference_pays['pays'].apply(standardiser_nom_pays)
reference_pays['pays_source_en_clean'] = reference_pays['pays_source_en'].apply(standardiser_nom_pays)

# Standardiser les noms dans dim_pays
dim_pays['Country_clean'] = dim_pays['Country'].apply(standardiser_nom_pays)

# Fusionner dim_pays avec reference_pays sur 'pays_clean'
dim_pays_enriched = dim_pays.merge(
    reference_pays,
    left_on='Country_clean',
    right_on='pays_clean',
    how='left'
)

# Si certains pays ne sont pas trouvés, essayer avec 'pays_source_en_clean'
pays_non_trouves = dim_pays_enriched[dim_pays_enriched['pays'].isna()]
if not pays_non_trouves.empty:
    dim_pays_restants = dim_pays[dim_pays['Country_clean'].isin(pays_non_trouves['Country_clean'])]
    dim_pays_restants_enriched = dim_pays_restants.merge(
        reference_pays,
        left_on='Country_clean',
        right_on='pays_source_en_clean',
        how='left'
    )
    # Mettre à jour dim_pays_enriched avec les résultats de la deuxième fusion
    for col in reference_pays.columns:
        if col in dim_pays_enriched.columns:
            dim_pays_enriched[col] = dim_pays_enriched[col].fillna(dim_pays_restants_enriched[col])

In [15]:
# --- 2. Nettoyer les doublons de colonnes ---
# Supprimer les colonnes inutiles (celles avec '_clean')
dim_pays_enriched = dim_pays_enriched.drop(columns=['pays_clean', 'pays_source_en_clean', 'Country_clean'], errors='ignore')

# Supprimer les doublons de colonnes (ex: 'pays', 'pays')
dim_pays_enriched = dim_pays_enriched.loc[:, ~dim_pays_enriched.columns.duplicated(keep='first')]

# Renommer les colonnes pour correspondre à votre structure
# Vérifier d'abord si les colonnes existent avant de les renommer
colonnes_a_renommer = {
    'pays': 'pays_fr',
    'pays_source_en': 'pays_en',
    'iso3': 'code_iso3',
    'region': 'region_oms',
}
# Appliquer le renommage uniquement si la colonne existe
dim_pays_enriched = dim_pays_enriched.rename(columns={k: v for k, v in colonnes_a_renommer.items() if k in dim_pays_enriched.columns})


# Ajouter les colonnes manquantes si elles n'existent pas
for colonne in ['pays_fr', 'pays_en', 'code_iso3', 'region_oms', 'continent', 'sous_region_onu']:
    if colonne not in dim_pays_enriched.columns:
        dim_pays_enriched[colonne] = np.nan

# Sélectionner uniquement les colonnes utiles
colonnes_utilisees = ['Country', 'Region', 'pays_fr', 'pays_en', 'code_iso3', 'region_oms', 'continent', 'sous_region_onu']
dim_pays_enriched = dim_pays_enriched[colonnes_utilisees]



In [16]:
# Dictionnaire des corrections pour les pays manquants (complété avec continent et sous_region_onu)
corrections_pays = {
    "china": {
        "pays_fr": "Chine",
        "pays_en": "China",
        "code_iso3": "CHN",
        "region_oms": "Western Pacific",
        "continent": "Asia",
        "sous_region_onu": "Eastern Asia"
    },
    "dominican republic": {
        "pays_fr": "République dominicaine",
        "pays_en": "Dominican Republic",
        "code_iso3": "DOM",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "egypt": {
        "pays_fr": "Égypte",
        "pays_en": "Egypt",
        "code_iso3": "EGY",
        "region_oms": "Eastern Mediterranean",
        "continent": "Africa",
        "sous_region_onu": "Northern Africa"
    },
    "equatorial guinea": {
        "pays_fr": "Guinée équatoriale",
        "pays_en": "Equatorial Guinea",
        "code_iso3": "GNQ",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Middle Africa"
    },
    "eritrea": {
        "pays_fr": "Érythrée",
        "pays_en": "Eritrea",
        "code_iso3": "ERI",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Eastern Africa"
    },
    "fiji": {
        "pays_fr": "Fidji",
        "pays_en": "Fiji",
        "code_iso3": "FJI",
        "region_oms": "Western Pacific",
        "continent": "Oceania",
        "sous_region_onu": "Melanesia"
    },
    "guinea": {
        "pays_fr": "Guinée",
        "pays_en": "Guinea",
        "code_iso3": "GIN",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Western Africa"
    },
    "guinea-bissau": {
        "pays_fr": "Guinée-Bissau",
        "pays_en": "Guinea-Bissau",
        "code_iso3": "GNB",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Western Africa"
    },
    "haiti": {
        "pays_fr": "Haïti",
        "pays_en": "Haiti",
        "code_iso3": "HTI",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "indonesia": {
        "pays_fr": "Indonésie",
        "pays_en": "Indonesia",
        "code_iso3": "IDN",
        "region_oms": "South-East Asia",
        "continent": "Asia",
        "sous_region_onu": "South-eastern Asia"
    },
    "jamaica": {
        "pays_fr": "Jamaïque",
        "pays_en": "Jamaica",
        "code_iso3": "JAM",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "libya": {
        "pays_fr": "Libye",
        "pays_en": "Libya",
        "code_iso3": "LBY",
        "region_oms": "Eastern Mediterranean",
        "continent": "Africa",
        "sous_region_onu": "Northern Africa"
    },
    "marshall islands": {
        "pays_fr": "Îles Marshall",
        "pays_en": "Marshall Islands",
        "code_iso3": "MHL",
        "region_oms": "Western Pacific",
        "continent": "Oceania",
        "sous_region_onu": "Micronesia"
    },
    "mauritania": {
        "pays_fr": "Mauritanie",
        "pays_en": "Mauritania",
        "code_iso3": "MRT",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Western Africa"
    },
    "mauritius": {
        "pays_fr": "Maurice",
        "pays_en": "Mauritius",
        "code_iso3": "MUS",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Eastern Africa"
    },
    "micronesia (federated states of)": {
        "pays_fr": "Micronésie",
        "pays_en": "Micronesia (Federated States of)",
        "code_iso3": "FSM",
        "region_oms": "Western Pacific",
        "continent": "Oceania",
        "sous_region_onu": "Micronesia"
    },
    "myanmar": {
        "pays_fr": "Myanmar (Birmanie)",
        "pays_en": "Myanmar",
        "code_iso3": "MMR",
        "region_oms": "South-East Asia",
        "continent": "Asia",
        "sous_region_onu": "South-eastern Asia"
    },
    "namibia": {
        "pays_fr": "Namibie",
        "pays_en": "Namibia",
        "code_iso3": "NAM",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Southern Africa"
    },
    "palau": {
        "pays_fr": "Palaos",
        "pays_en": "Palau",
        "code_iso3": "PLW",
        "region_oms": "Western Pacific",
        "continent": "Oceania",
        "sous_region_onu": "Micronesia"
    },
    "papua new guinea": {
        "pays_fr": "Papouasie-Nouvelle-Guinée",
        "pays_en": "Papua New Guinea",
        "code_iso3": "PNG",
        "region_oms": "Western Pacific",
        "continent": "Oceania",
        "sous_region_onu": "Melanesia"
    },
    "saint kitts and nevis": {
        "pays_fr": "Saint-Christophe-et-Niévès",
        "pays_en": "Saint Kitts and Nevis",
        "code_iso3": "KNA",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "saint lucia": {
        "pays_fr": "Sainte-Lucie",
        "pays_en": "Saint Lucia",
        "code_iso3": "LCA",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "saint vincent and the grenadines": {
        "pays_fr": "Saint-Vincent-et-les-Grenadines",
        "pays_en": "Saint Vincent and the Grenadines",
        "code_iso3": "VCT",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "sao tome and principe": {
        "pays_fr": "Sao Tomé-et-Principe",
        "pays_en": "Sao Tome and Principe",
        "code_iso3": "STP",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Middle Africa"
    },
    "saudi arabia": {
        "pays_fr": "Arabie saoudite",
        "pays_en": "Saudi Arabia",
        "code_iso3": "SAU",
        "region_oms": "Eastern Mediterranean",
        "continent": "Asia",
        "sous_region_onu": "Western Asia"
    },
    "senegal": {
        "pays_fr": "Sénégal",
        "pays_en": "Senegal",
        "code_iso3": "SEN",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Western Africa"
    },
    "solomon islands": {
        "pays_fr": "Îles Salomon",
        "pays_en": "Solomon Islands",
        "code_iso3": "SLB",
        "region_oms": "Western Pacific",
        "continent": "Oceania",
        "sous_region_onu": "Melanesia"
    },
    "somalia": {
        "pays_fr": "Somalie",
        "pays_en": "Somalia",
        "code_iso3": "SOM",
        "region_oms": "Eastern Mediterranean",
        "continent": "Africa",
        "sous_region_onu": "Eastern Africa"
    },
    "south sudan": {
        "pays_fr": "Soudan du Sud",
        "pays_en": "South Sudan",
        "code_iso3": "SSD",
        "region_oms": "Africa",
        "continent": "Africa",
        "sous_region_onu": "Eastern Africa"
    },
    "sudan": {
        "pays_fr": "Soudan",
        "pays_en": "Sudan",
        "code_iso3": "SDN",
        "region_oms": "Eastern Mediterranean",
        "continent": "Africa",
        "sous_region_onu": "Northern Africa"
    },
    "syrian arab republic": {
        "pays_fr": "Syrie",
        "pays_en": "Syrian Arab Republic",
        "code_iso3": "SYR",
        "region_oms": "Eastern Mediterranean",
        "continent": "Asia",
        "sous_region_onu": "Western Asia"
    },
    "thailand": {
        "pays_fr": "Thaïlande",
        "pays_en": "Thailand",
        "code_iso3": "THA",
        "region_oms": "South-East Asia",
        "continent": "Asia",
        "sous_region_onu": "South-eastern Asia"
    },
    "timor-leste": {
        "pays_fr": "Timor oriental",
        "pays_en": "Timor-Leste",
        "code_iso3": "TLS",
        "region_oms": "South-East Asia",
        "continent": "Asia",
        "sous_region_onu": "South-eastern Asia"
    },
    "trinidad and tobago": {
        "pays_fr": "Trinité-et-Tobago",
        "pays_en": "Trinidad and Tobago",
        "code_iso3": "TTO",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "Caribbean"
    },
    "turkey": {
        "pays_fr": "Turquie",
        "pays_en": "Turkey",
        "code_iso3": "TUR",
        "region_oms": "Europe",
        "continent": "Asia",
        "sous_region_onu": "Western Asia"
    },
    "united arab emirates": {
        "pays_fr": "Émirats arabes unis",
        "pays_en": "United Arab Emirates",
        "code_iso3": "ARE",
        "region_oms": "Eastern Mediterranean",
        "continent": "Asia",
        "sous_region_onu": "Western Asia"
    },
    "venezuela (bolivarian republic of)": {
        "pays_fr": "Venezuela",
        "pays_en": "Venezuela (Bolivarian Republic of)",
        "code_iso3": "VEN",
        "region_oms": "Americas",
        "continent": "Americas",
        "sous_region_onu": "South America"
    },
    "viet nam": {
        "pays_fr": "Viêt Nam",
        "pays_en": "Viet Nam",
        "code_iso3": "VNM",
        "region_oms": "Western Pacific",
        "continent": "Asia",
        "sous_region_onu": "South-eastern Asia"
    },
    "yemen": {
        "pays_fr": "Yémen",
        "pays_en": "Yemen",
        "code_iso3": "YEM",
        "region_oms": "Eastern Mediterranean",
        "continent": "Asia",
        "sous_region_onu": "Western Asia"
    }
}


# Appliquer les corrections aux lignes manquantes dans dim_pays_enriched
for index, row in dim_pays_enriched.iterrows():
    pays_clean = row['Country'].lower().strip()
    if pays_clean in corrections_pays:
        metadata = corrections_pays[pays_clean]
        for colonne, valeur in metadata.items():
            if pd.isna(row[colonne]):
                dim_pays_enriched.at[index, colonne] = valeur

# Remplir les valeurs manquantes pour continent et sous_region_onu
# en utilisant les valeurs de reference_pays (si disponibles)
for index, row in dim_pays_enriched.iterrows():
    if pd.isna(row['continent']):
        # Essayer de trouver le pays dans reference_pays
        pays_clean = row['Country'].lower().strip()
        ref_row = reference_pays[reference_pays['pays_clean'] == pays_clean]
        if not ref_row.empty:
            dim_pays_enriched.at[index, 'continent'] = ref_row.iloc[0]['continent']
            dim_pays_enriched.at[index, 'sous_region_onu'] = ref_row.iloc[0]['sous_region_onu']
        else:
            # Utiliser les valeurs par défaut basées sur region_oms
            if row['region_oms'] == 'Africa':
                dim_pays_enriched.at[index, 'continent'] = 'Africa'
                dim_pays_enriched.at[index, 'sous_region_onu'] = 'Eastern Africa'  # Valeur par défaut
            elif row['region_oms'] == 'Americas':
                dim_pays_enriched.at[index, 'continent'] = 'Americas'
                dim_pays_enriched.at[index, 'sous_region_onu'] = 'Caribbean'  # Valeur par défaut
            elif row['region_oms'] == 'Eastern Mediterranean':
                dim_pays_enriched.at[index, 'continent'] = 'Asia'
                dim_pays_enriched.at[index, 'sous_region_onu'] = 'Western Asia'  # Valeur par défaut
            elif row['region_oms'] == 'Europe':
                dim_pays_enriched.at[index, 'continent'] = 'Europe'
                dim_pays_enriched.at[index, 'sous_region_onu'] = 'Western Europe'  # Valeur par défaut
            elif row['region_oms'] == 'South-East Asia':
                dim_pays_enriched.at[index, 'continent'] = 'Asia'
                dim_pays_enriched.at[index, 'sous_region_onu'] = 'South-eastern Asia'  # Valeur par défaut
            elif row['region_oms'] == 'Western Pacific':
                dim_pays_enriched.at[index, 'continent'] = 'Oceania'
                dim_pays_enriched.at[index, 'sous_region_onu'] = 'Melanesia'  # Valeur par défaut

print("✅ Corrections appliquées aux pays manquants et valeurs manquantes remplies !")

✅ Corrections appliquées aux pays manquants et valeurs manquantes remplies !


In [17]:
# Supprimer les colonnes inutiles (celles avec '_clean')
dim_pays_enriched = dim_pays_enriched.drop(columns=['pays_clean', 'pays_source_en_clean', 'Country_clean'], errors='ignore')

# Supprimer les doublons de colonnes (ex: pays_fr, pays_fr)
dim_pays_enriched = dim_pays_enriched.loc[:, ~dim_pays_enriched.columns.duplicated(keep='first')]

# Renommer les colonnes pour correspondre à votre structure
dim_pays_enriched = dim_pays_enriched.rename(columns={
    'pays': 'pays_fr',
    'pays_source_en': 'pays_en',
    'iso3': 'code_iso3',
    'region': 'region_oms',
})

# Sélectionner uniquement les colonnes utiles
colonnes_utilisees = ['Country', 'Region', 'pays_fr', 'pays_en', 'code_iso3', 'region_oms', 'continent', 'sous_region_onu']
dim_pays_enriched = dim_pays_enriched[colonnes_utilisees]

In [18]:
# Vérifier les doublons de colonnes
print("Colonnes après nettoyage :", dim_pays_enriched.columns.tolist())

# Vérifier les valeurs manquantes
print("\nValeurs manquantes après corrections :")
print(dim_pays_enriched.isnull().sum())

# Afficher un échantillon des données
print("\nÉchantillon des données enrichies :")
print(dim_pays_enriched.head(10))

Colonnes après nettoyage : ['Country', 'Region', 'pays_fr', 'pays_en', 'code_iso3', 'region_oms', 'continent', 'sous_region_onu']

Valeurs manquantes après corrections :
Country             0
Region              0
pays_fr            36
pays_en            36
code_iso3          36
region_oms         36
continent          36
sous_region_onu    36
dtype: int64

Échantillon des données enrichies :
      Country                 Region      pays_fr     pays_en code_iso3  \
0     albania                 Europe      Albanie     Albania       ALB   
1     andorra                 Europe      Andorre     Andorra       AND   
2     armenia                 Europe      Arménie     Armenia       ARM   
3   australia        Western Pacific    Australie   Australia       AUS   
4     austria                 Europe     Autriche     Austria       AUT   
5  azerbaijan                 Europe  Azerbaïdjan  Azerbaijan       AZE   
6     bahrain  Eastern Mediterranean      Bahreïn     Bahrain       BHR   
7  b

In [19]:
# --- 5. Remplacer dim_pays par dim_pays_enriched ---
# Remplacer simplement la variable dim_pays par dim_pays_enriched
dim_pays = dim_pays_enriched

# --- 5. Export en CSV ---
# Exporter les tables dimensionnelles
dim_pays.to_csv("data/processed/dim_pays.csv", index=False)
dim_annee.to_csv("data/processed/dim_annee.csv", index=False)
dim_indicateur.to_csv("data/processed/dim_indicateur.csv", index=False)

# Exporter la table de faits
table_faits.to_csv("data/processed/faits_indicateurs.csv", index=False)

print("✅ Export des fichiers CSV terminé !")

✅ Export des fichiers CSV terminé !
